In [1]:
!pip -q install numpy pandas faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 10.9 MB/s eta 0:00:00


In [2]:
runbooks = [
("""RUNBOOK: SAP Login Failures
Symptoms: Users see 'SAML assertion invalid' or repeated login redirects.
Checks:
1) Verify IdP certificate validity and rotation date.
2) Check system clock skew on app servers (< 2 minutes).
3) Validate SAML audience/issuer values match configuration.
Fix:
- Update IdP certificate, restart auth service, re-test SSO.
Prevention:
- Monitor certificate expiry and enable alerting 30 days prior.""", "RB_SSO_001"),

("""RUNBOOK: High CPU on Application Pods
Symptoms: Response latency increases, CPU throttling, autoscaler triggers.
Checks:
1) Identify hottest endpoints and recent deployments.
2) Inspect thread dumps and GC logs if JVM.
3) Verify resource limits/requests and HPA settings.
Fix:
- Roll back recent changes, increase CPU limits temporarily, tune GC or caching.
Prevention:
- Load testing before release; dashboards for p95 latency and CPU.""", "RB_PERF_002"),

("""RUNBOOK: Kafka Consumer Lag
Symptoms: Lag increases, delayed processing, timeouts downstream.
Checks:
1) Confirm broker health and partition leadership.
2) Inspect consumer group rebalances and commit frequency.
3) Verify message size and processing time spikes.
Fix:
- Scale consumer replicas, optimize processing, increase max.poll.interval.ms if needed.
Prevention:
- Alerts for lag threshold and backpressure handling.""", "RB_KAFKA_003"),

("""RUNBOOK: Database Connection Pool Exhaustion
Symptoms: 'Timeout waiting for connection', error rate spikes, slow queries.
Checks:
1) Look for long-running queries and lock waits.
2) Verify pool size vs concurrency, and timeouts.
3) Check DB CPU/IO and recent schema changes.
Fix:
- Optimize slow queries, increase pool size carefully, add indexes, reduce chatty queries.
Prevention:
- Query monitoring; capacity planning and load tests.""", "RB_DB_004"),
]
len(runbooks)

4

In [3]:
incidents = [
("""Timestamp: 2026-02-18 10:14:22
Service: auth-gateway
Error: SAML assertion invalid; audience mismatch
Impact: ~40% users cannot login; repeated redirects observed
Notes: IdP certificate rotated yesterday""", "login_issue"),

("""Timestamp: 2026-02-20 16:05:11
Service: stream-processor
Error: Consumer lag increasing rapidly, commit timeouts
Impact: Orders delayed by 25 minutes
Notes: Spike in message size after new feature rollout""", "kafka_lag"),

("""Timestamp: 2026-02-22 09:41:03
Service: checkout-api
Error: Timeout waiting for connection from pool
Impact: 12% requests failing; p95 latency 2.8s
Notes: DB shows lock waits on orders table""", "db_pool"),

("""Timestamp: 2026-02-24 13:22:55
Service: pricing-service
Error: CPU throttling; p95 latency increased
Impact: API timeouts during peak traffic
Notes: Deployed caching change 30 minutes ago""", "high_cpu"),
]
len(incidents)

4

In [4]:
def chunk_text(text, chunk_words=220, overlap_words=40):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_words, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = end - overlap_words
    return chunks

chunks = []
chunk_meta = []

for rb_text, rb_id in runbooks:
    rb_chunks = chunk_text(rb_text, chunk_words=220, overlap_words=40)
    for i, ch in enumerate(rb_chunks):
        chunks.append(ch)
        chunk_meta.append({"runbook_id": rb_id, "chunk_id": f"{rb_id}_c{i}"})

len(chunks), chunk_meta[0]

(4, {'runbook_id': 'RB_SSO_001', 'chunk_id': 'RB_SSO_001_c0'})

In [5]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")  # fast + good baseline

emb = embedder.encode(chunks, convert_to_numpy=True, normalize_embeddings=True)

dim = emb.shape[1]
index = faiss.IndexFlatIP(dim)  # cosine-like similarity (because normalized)
index.add(emb)

index.ntotal

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

4

In [6]:
def retrieve_context(query, top_k=3):
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, ids = index.search(q_emb, top_k)
    results = []
    for score, idx_ in zip(scores[0], ids[0]):
        results.append({
            "score": float(score),
            "text": chunks[idx_],
            "meta": chunk_meta[idx_]
        })
    return results

test_query = "SAML assertion invalid audience mismatch after certificate rotation"
hits = retrieve_context(test_query, top_k=3)

for h in hits:
    print(h["meta"], "score=", round(h["score"], 3))

{'runbook_id': 'RB_SSO_001', 'chunk_id': 'RB_SSO_001_c0'} score= 0.565
{'runbook_id': 'RB_KAFKA_003', 'chunk_id': 'RB_KAFKA_003_c0'} score= 0.148
{'runbook_id': 'RB_PERF_002', 'chunk_id': 'RB_PERF_002_c0'} score= 0.012


In [7]:
def grounded_recommendations(log_text, top_k=3):
    ctx = retrieve_context(log_text, top_k=top_k)
    out = []
    out.append("INCIDENT (input):\n" + log_text)
    out.append("\nTOP RUNBOOK MATCHES (retrieved):")
    for c in ctx:
        out.append(f"- {c['meta']['chunk_id']} (runbook={c['meta']['runbook_id']}, score={c['score']:.3f})")
    out.append("\nNEXT: Use an LLM to summarize these steps with citations.")
    return "\n".join(out)

print(grounded_recommendations(incidents[0][0], top_k=3))

INCIDENT (input):
Timestamp: 2026-02-18 10:14:22
Service: auth-gateway
Error: SAML assertion invalid; audience mismatch
Impact: ~40% users cannot login; repeated redirects observed
Notes: IdP certificate rotated yesterday

TOP RUNBOOK MATCHES (retrieved):
- RB_SSO_001_c0 (runbook=RB_SSO_001, score=0.671)
- RB_KAFKA_003_c0 (runbook=RB_KAFKA_003, score=0.205)
- RB_DB_004_c0 (runbook=RB_DB_004, score=0.118)

NEXT: Use an LLM to summarize these steps with citations.


In [8]:
tests = [
("Why are users failing login with SAML assertion invalid?", incidents[0][0]),
("Why is consumer lag increasing?", incidents[1][0]),
("Why are DB connection timeouts happening?", incidents[2][0]),
("Why is CPU throttling causing latency?", incidents[3][0]),
]

for q, log in tests:
    hits = retrieve_context(q + "\n" + log, top_k=2)
    print("\nQUESTION:", q)
    print("Retrieved:", [h["meta"]["runbook_id"] for h in hits])


QUESTION: Why are users failing login with SAML assertion invalid?
Retrieved: ['RB_SSO_001', 'RB_KAFKA_003']

QUESTION: Why is consumer lag increasing?
Retrieved: ['RB_KAFKA_003', 'RB_PERF_002']

QUESTION: Why are DB connection timeouts happening?
Retrieved: ['RB_DB_004', 'RB_KAFKA_003']

QUESTION: Why is CPU throttling causing latency?
Retrieved: ['RB_PERF_002', 'RB_KAFKA_003']
